# UNIVERSIDAD NACIONAL DE CHIMBORAZO
**Cristian Alvear**

## Caso de estudio: clasificación de flores Iris con k-NN

### 1. Cargar el archivo Iris

In [3]:
import numpy as np

# El archivo tiene una columna Id al inicio, por eso las características están en columnas 1 a 4
X = np.genfromtxt(
    "Iris.csv",
    delimiter=",",
    skip_header=1,
    usecols=(1, 2, 3, 4),
    dtype=float
)

y = np.genfromtxt(
    "Iris.csv",
    delimiter=",",
    skip_header=1,
    usecols=5,
    dtype=str
)

print("Primeras 5 filas de X:")
print(X[:5])
print("\nPrimeras 5 etiquetas de y:")
print(y[:5])
print("\nForma de X:", X.shape)
print("Forma de y:", y.shape)
print("\nCantidad total de flores:", X.shape[0])
print("Cantidad de características por flor:", X.shape[1])

Primeras 5 filas de X:
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]

Primeras 5 etiquetas de y:
['Iris-setosa' 'Iris-setosa' 'Iris-setosa' 'Iris-setosa' 'Iris-setosa']

Forma de X: (150, 4)
Forma de y: (150,)

Cantidad total de flores: 150
Cantidad de características por flor: 4


**Explicación:**

- `X` contiene las cuatro características numéricas de cada flor: largo del sépalo, ancho del sépalo, largo del pétalo y ancho del pétalo. Es la entrada del modelo.
- `y` contiene la especie correspondiente a cada flor (Iris-setosa, Iris-versicolor o Iris-virginica). Es la etiqueta o clase que el algoritmo debe aprender a predecir.

### 2. Revisar las clases del dataset

In [ ]:
clases, conteos = np.unique(y, return_counts=True)

print("Especies disponibles:", clases)
print("\nCantidad de datos por especie:")
for clase, conteo in zip(clases, conteos):
    print(f"  {clase}: {conteo}")
print("\nNúmero total de clases:", len(clases))

**Interpretación:**

Las tres clases tienen exactamente 50 registros cada una, lo que indica que el dataset está balanceado. Esto es conveniente porque el algoritmo no tendrá tendencia a favorecer ninguna especie por tener más ejemplos.

Conocer las clases antes de aplicar un clasificador es importante para saber cuántas categorías debe distinguir el modelo y para detectar posibles desequilibrios que puedan afectar el rendimiento.

### 3. Separar datos de entrenamiento y prueba

In [ ]:
X_train = X[:120]
X_test  = X[120:]
y_train = y[:120]
y_test  = y[120:]

print("Datos de entrenamiento:", X_train.shape[0])
print("Datos de prueba:", X_test.shape[0])
print("\nForma de X_train:", X_train.shape)
print("Forma de X_test: ", X_test.shape)
print("Forma de y_train:", y_train.shape)
print("Forma de y_test: ", y_test.shape)

**Explicación:**

Separar los datos es necesario para evaluar el modelo de forma honesta. Si se entrenara y evaluara con los mismos datos, el algoritmo podría simplemente memorizar las respuestas y aparentar un rendimiento perfecto sin haber aprendido a generalizar a nuevos casos.

### 4. Seleccionar una flor para clasificar

In [ ]:
flor = X_test[0]
clase_real = y_test[0]

print("Valores de la flor seleccionada:", flor)
print("Especie real:", clase_real)

**Explicación:**

- La flor seleccionada es el primer registro del conjunto de prueba. Sus cuatro valores representan las medidas de sus sépalos y pétalos.
- Conocer la clase real sirve para verificar después si el algoritmo predijo correctamente.
- El algoritmo no debe usar la clase real durante la predicción porque en la práctica ese dato no existe cuando se tiene una flor nueva. Usarlo sería hacer trampa y el resultado no sería válido.

### 5. Calcular las distancias

In [ ]:
diferencias = X_train - flor
cuadrados   = diferencias ** 2
sumas       = np.sum(cuadrados, axis=1)
distancias  = np.sqrt(sumas)

print("Primeras 10 distancias:")
print(distancias[:10])
print("\nCantidad total de distancias:", len(distancias))

**Interpretación:**

- Una distancia pequeña indica que esa flor de entrenamiento es muy similar a la flor que se quiere clasificar.
- Una distancia grande indica que esa flor es muy diferente.
- El algoritmo k-NN necesita calcular distancias porque su criterio de clasificación es la similitud: asume que flores con características parecidas pertenecen a la misma especie.

### 6. Encontrar los vecinos más cercanos

In [ ]:
k = 5
indices_vecinos = np.argpartition(distancias, k)[:k]

print("Posiciones de los 5 vecinos más cercanos:", indices_vecinos)
print("Distancias de esos vecinos:", distancias[indices_vecinos])
print("Especies de esos vecinos:", y_train[indices_vecinos])

**Explicación:**

- `argpartition(distancias, k)` reorganiza los índices del arreglo de forma que los `k` valores más pequeños queden en las primeras posiciones, sin ordenar el resto. Esto es más eficiente que ordenar todo el arreglo.
- Sirve para encontrar vecinos cercanos porque solo se necesitan los índices de los menores valores, no el orden completo.
- `argpartition` es más rápido que `argsort` cuando solo interesa un subconjunto de los valores mínimos, porque no ordena los elementos restantes.
- No es necesario ordenar todas las distancias porque el algoritmo solo usa los `k` vecinos más próximos; el orden del resto no afecta la predicción.

### 7. Realizar la votación

In [ ]:
especies_vecinos = y_train[indices_vecinos]
clases_unicas, votos = np.unique(especies_vecinos, return_counts=True)

prediccion = clases_unicas[np.argmax(votos)]

print("Especies de los vecinos:", especies_vecinos)
print("Especie predicha:", prediccion)
print("Especie real:", clase_real)
print("Predicción correcta:", prediccion == clase_real)

**Explicación:**

- La especie con más votos entre los vecinos es la predicción del algoritmo.
- La votación es el mecanismo central de k-NN: la clase que aparece más veces entre los vecinos más cercanos se considera la más probable para la nueva flor.
- Si dos especies tienen la misma cantidad de votos hay un empate. En ese caso el resultado depende de cómo esté implementado el desempate; una opción común es reducir k en uno o usar la distancia para decidir.

### 8. Crear una función para el algoritmo k-NN

In [ ]:
def knn_predict(X_train, y_train, nueva_flor, k):
    distancias = np.sqrt(np.sum((X_train - nueva_flor) ** 2, axis=1))
    indices_vecinos = np.argpartition(distancias, k)[:k]
    especies_vecinos = y_train[indices_vecinos]
    clases_unicas, votos = np.unique(especies_vecinos, return_counts=True)
    return clases_unicas[np.argmax(votos)]


# Prueba con la primera flor del conjunto de prueba
pred = knn_predict(X_train, y_train, X_test[0], k=5)
print("Predicción:", pred)
print("Clase real:", y_test[0])

**Explicación:**

- La función recibe `X_train` (características de entrenamiento), `y_train` (etiquetas de entrenamiento), `nueva_flor` (la flor a clasificar) y `k` (número de vecinos).
- Devuelve una cadena de texto con el nombre de la especie predicha.
- Al encapsular el proceso en una función es posible reutilizarlo fácilmente para clasificar cualquier flor con cualquier valor de k.

### 9. Clasificar todas las flores de prueba

In [ ]:
predicciones = np.array([knn_predict(X_train, y_train, flor, k=5) for flor in X_test])

correctas   = np.sum(predicciones == y_test)
incorrectas = np.sum(predicciones != y_test)
porcentaje  = (correctas / len(y_test)) * 100

print("Flores clasificadas correctamente:", correctas)
print("Flores clasificadas incorrectamente:", incorrectas)
print("Porcentaje de acierto:", round(porcentaje, 2), "%")

**Interpretación:**

Un porcentaje de acierto cercano o igual al 100% indica que el algoritmo funciona bien para este dataset. Para mejorar el rendimiento en casos más difíciles se podría normalizar las características antes de calcular distancias, o ajustar el valor de k de forma más sistemática.

### 10. Probar diferentes valores de k

In [ ]:
valores_k = [1, 3, 5, 7, 9]

print(f"{'k':>4}  {'Acierto (%)':>12}")
print("-" * 20)

mejores = []
for k_val in valores_k:
    preds = np.array([knn_predict(X_train, y_train, flor, k=k_val) for flor in X_test])
    acc = (np.sum(preds == y_test) / len(y_test)) * 100
    mejores.append(acc)
    print(f"{k_val:>4}  {acc:>11.2f}%")

mejor_k = valores_k[np.argmax(mejores)]
print("\nMejor valor de k:", mejor_k)

**Interpretación:**

- El porcentaje de acierto puede variar al cambiar k, aunque en este dataset las diferencias suelen ser pequeñas.
- Probar varios valores de k es importante porque no existe un valor óptimo único; depende de los datos.
- Si k es muy pequeño (k=1), el modelo es muy sensible al ruido y puede equivocarse por un dato atípico.
- Si k es muy grande, el modelo puede volverse demasiado general e ignorar patrones locales relevantes.

### 11. Análisis del uso de NumPy en el algoritmo

**Uso de NumPy dentro del algoritmo k-NN:**

1. **Indexación:** se usó para acceder a filas específicas, como `X_test[0]` para seleccionar una flor o `y_train[indices_vecinos]` para obtener las especies de los vecinos.
2. **Slicing:** se usó para dividir el dataset en entrenamiento y prueba con `X[:120]` y `X[120:]`.
3. **axis:** se usó `axis=1` en `np.sum` para sumar las diferencias al cuadrado por fila, es decir, por cada flor de entrenamiento.
4. **argpartition:** se usó para encontrar las posiciones de los k vecinos más cercanos sin ordenar todo el arreglo de distancias.
5. **argmax:** se usó para identificar la especie con mayor cantidad de votos entre los vecinos.
6. **Operaciones vectorizadas:** la resta `X_train - nueva_flor` se aplicó sobre toda la matriz a la vez, sin necesidad de un ciclo. Lo mismo ocurre con la potencia y la raíz cuadrada.
7. **Comparación con listas tradicionales:** con listas sería necesario usar ciclos explícitos para cada operación matemática, lo que hace el código más largo y lento. NumPy realiza esas mismas operaciones de forma directa y eficiente.

### Reflexión final

Implementar el algoritmo k-NN desde cero con NumPy permitió entender cómo funciona internamente un clasificador, sin depender de librerías que ocultan los pasos. Se pudo comprobar que NumPy es una herramienta eficiente para trabajar con datos numéricos organizados en arreglos, ya que permite realizar cálculos sobre matrices completas en una sola instrucción. El uso de `argpartition` resultó especialmente útil porque evita ordenar todas las distancias cuando solo se necesitan las k más pequeñas, lo que mejora el rendimiento del algoritmo. Trabajar paso a paso facilitó identificar con claridad qué hace cada parte del proceso: el cálculo de distancias, la selección de vecinos y la votación. El algoritmo k-NN tiene aplicaciones en problemas reales como el diagnóstico médico, la detección de fraudes, los sistemas de recomendación y el reconocimiento de imágenes. La principal dificultad fue entender el manejo de índices al combinar `argpartition` con la indexación de arreglos, ya que es fácil confundir las posiciones con los valores.